# 4 - GEODE: Building the Oracle

GEODE turns a document into a trained, self-corrected store: ingest, a propose -> diagnose -> repair loop that drops geometrically-implausible triples, then training and calibration. In this notebook we **run** a real (small) build, inspect what it produced, then look at the full production store.

## Run GEODE on a small document
A well-formed markdown table is enough. `build_rag_store` runs the `GeodeLoop` self-correction orchestrator and a geometry-supervised embedding loop, then trains the production store (small config here so it runs in seconds on CPU).

In [ ]:
import tempfile, torch
from pathlib import Path
from knowlytix.knowledge.geode import build_rag_store, make_default_trainer
from knowlytix.knowledge.config import DocGMSConfig

doc = Path(tempfile.mktemp(suffix='.md'))
doc.write_text('''# Bank Fee Schedule\n\n## Fee Schedule\n\n| product | fee_amount | type |\n| --- | --- | --- |\n| overdraft | 35.00 | per_occurrence |\n| wire_international | 45.00 | per_transaction |\n| stop_payment | 30.00 | per_request |\n''')

cfg = DocGMSConfig(store_path=tempfile.mkdtemp(), ingest_mode='regex')
cfg.train.epochs = 60                       # small, for a fast notebook build
res = build_rag_store(doc, cfg, device=torch.device('cpu'),
                      geode_trainer=make_default_trainer(torch.device('cpu'), epochs=60),
                      max_iters=3)
print(f'converged={res.converged} iters={res.iterations} '
      f'entities={res.n_entities} triples={res.n_triples} enm={res.n_enm} '
      f'corrections={len(res.corrections)}')

The build returns a `GMSExpertStore` trained on the surviving triples. We query it directly --- the same primitives as Chapter 3, now on a store we just built.

In [ ]:
print('asserted edges:', res.store.query_triples(head='overdraft'))
print('score_triple(overdraft, has_fee_amount, 35.0) =',
      round(res.store.score_triple('overdraft', 'has_fee_amount', '35.0'), 3))

## What the loop does
The `GeodeLoop` proposes the regex-extracted triples, diagnoses the ones that sit implausibly far on the manifold (or contradict an established fact) and repairs or drops them, recording every correction. On a clean table there is little to correct; on noisy prose the loop removes the extraction errors an oracle must not inherit. A geometry-supervised embedding loop also tunes the encoder to the document's vocabulary, so later queries bind colloquial wording to the right entity --- the ingestion improvement. (Mechanics live in the GMS substrate.)

## The full production store
The book's policy store is the same build at scale. It covers every policy, including the prose ones (filing windows, closure notice, escalation windows) that a fee-table-only ingest would miss.

In [ ]:
import torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore

def _find_store(name="gms_policy_store_geode"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        c = p / "beyond-prompt-and-pray" / "code" / "data" / name
        if c.exists():
            return c
    raise FileNotFoundError(name + " not found; run scripts/build_geode_rag_store.py")

STORE = _find_store()
store = GMSExpertStore(DocGMSConfig(store_path=str(STORE), ingest_mode="regex"),
                       device=torch.device("cpu"))
assert store.load(), "store failed to load"
print("loaded store:", len(store.adapter.relation_to_idx), "relations,",
      len(store.adapter.entity_to_idx), "entities")

In [ ]:
for r in ['has_fee_amount', 'has_filing_window_days',
          'has_bank_initiated_notice_days', 'has_escalation_window_days']:
    print(f'{r}:', store.query_triples(relation=r)[:1])

## Self-check

In [ ]:
assert res.converged and res.n_triples > 0          # we built a real store
assert res.store.query_triples(head='overdraft')    # and can query it
assert store.query_triples(relation='has_filing_window_days')  # prose policy captured
print('OK - GEODE: building the oracle')